In [2]:
from Bio import Entrez
import pandas as pd
import numpy as np
import os
import json
import time
import regex as re
import xmltodict, json
# ElementTree
import xml.etree.ElementTree as ET
from dataclasses import dataclass

In [3]:
def search_pubmed(query, retmax=1000):
    Entrez.email = "c.g.a.viviers@tue.nl"  # Always include your email
    handle = Entrez.esearch(db="pubmed", term=query, retmax=retmax, retmode="xml")
    record = Entrez.read(handle)
    return record

def fetch_pubmed_data_per_id(pubmed_id):
    Entrez.email = "c.g.a.viviers@tue.nl"  # Always include your email
    handle = Entrez.efetch(db="pubmed", id=pubmed_id, rettype="xml")
    record = Entrez.read(handle)
    return record

def fetch_pubmed_data_given_ids(id_list):
    Entrez.email = "c.g.a.viviers@tue.nl"
    ids = ','.join(id_list)
    handle = Entrez.efetch(db='pubmed', retmode='xml', id=ids)
    results = Entrez.read(handle)
    return results

In [5]:
# data class for papers

@dataclass
class Paper:
    id: str
    title: str
    abstract: str
    authors: list
    journal: str
    publication_date: str
    doi: str
    keywords: list
    language_list: list
    embedding: np.array

    def __init__(self, id, title, abstract, authors, journal, publication_year, publication_month, publication_day, doi, keywords, language_list, embedding= None):
        self.id = id
        self.title = title
        self.abstract = abstract
        self.authors = authors
        self.journal = journal
        self.publication_year = publication_year
        self.publication_month = publication_month
        self.publication_day = publication_day
        self.doi = doi
        self.keywords = keywords
        self.language_list = language_list
        self.embedding = embedding

    def __str__(self):
        return f"Paper(id={self.id}, title={self.title}, abstract={self.abstract}, authors={self.authors}, journal={self.journal}, publication_year={self.publication_year}, publication_month={self.publication_month}, publication_day={self.publication_day}, doi={self.doi}, keywords={self.keywords}, language_list={self.language_list}, embedding={self.embedding})"
    
    def __repr__(self):
        return f"Paper(id={self.id}, title={self.title}, abstract={self.abstract}, authors={self.authors}, journal={self.journal}, publication_year={self.publication_year}, publication_month={self.publication_month}, publication_day={self.publication_day}, doi={self.doi}, keywords={self.keywords}, language_list={self.language_list}, embedding={self.embedding})"
    
    def save_to_json(self, filename):
        # save the paper to a json file in a nice format
        with open(filename, 'w') as f:
            json.dump(self.__dict__, f, indent=4)

    def load_from_json(self, filename):
        with open(filename, 'r') as f:
            data = json.load(f)
            return Paper(**data)
        

In [6]:
# E-Fetch runs with a maximum of ten thousand studies, we need to separate our IdList in chunks 
def fetch_pubmed_data_given_ids_in_chunks(id_list, chunk_size=10000, output_path=None):
    papers_list = []
    for i in range(0, len(id_list), chunk_size):

        chunk = id_list[i:i+chunk_size]
        papers = fetch_pubmed_data_given_ids(chunk)

        # store all the papers in a class
        for i, paper in enumerate (papers['PubmedArticle']):

            medline_citation = paper['MedlineCitation']

            article = medline_citation.get('Article', {})
            journal_info = article.get('Journal', {})
            journal_issue = journal_info.get('JournalIssue', {})
            pub_date = journal_issue.get('PubDate', {})

            # Extract the data from the XML
            id = medline_citation.get('PMID', 'Not Available')
            title = article.get('ArticleTitle', 'Not Available')
            abstract = article.get('Abstract', {}).get('AbstractText', 'Not Available')
            author_list = article.get('AuthorList', [])
            if isinstance(author_list, dict):
                authors = author_list.get('Author', [])
            else:
                authors = author_list

            authors = [f"{author.get('LastName', 'Not Available')} {author.get('Initials', 'Not Available')}" 
                    for author in authors if isinstance(author, dict)]

            journal = journal_info.get('Title', 'Not Available')
            publication_year = pub_date.get('Year', 'Not Available')
            publication_month = pub_date.get('Month', 'Not Available')
            publication_day = pub_date.get('Day', 'Not Available')
            doi = article.get('ELocationID', 'Not Available')
            keyword_list = medline_citation.get('KeywordList', [])
            if keyword_list and isinstance(keyword_list[0], dict):
                keywords = keyword_list[0].get('Keyword', ['Not Available'])
            else:
                keywords = ['Not Available']
            language_list = article.get('Language', ['Not Available'])

            # Create a Paper object
            paper_object = Paper(id, title, abstract, authors, journal, publication_year, publication_month, publication_day, doi, keywords, language_list)

            # Append the paper object to the list
            papers_list.append(paper_object)

            if output_path:
                if not os.path.exists(output_path):
                    os.makedirs(output_path, exist_ok=True)

                # check if the file already exists and that the id is writable
                filename = f"{output_path}/{id}.json"
                if not os.path.exists(filename) and id.isdigit():
                    paper_object.save_to_json(filename)

    return papers_list


In [7]:
# List of journals to include
journals_to_include = [
    "ACS Applied Materials & Interfaces",
    "ACS Nano",
    "Advanced Functional Materials",
    "Advanced Materials",
    "Angewandte Chemie",
    "Biology and Medicine",
    "Biomaterials",
    "Cell",
    "Clinical Cancer Research",
    "Frontiers in Nanotechnology",
    "Immunity",
    "International Journal of Nanomedicine",
    "Journal of Controlled Release",
    "Journal of Materials Chemistry B",
    "Matter",
    "Molecular Therapy",
    "Nano Letters",
    "Nano Micro Small",
    "Nano Research",
    "Nanomedicine",
    "Nanomedicine: Nanotechnology",
    "Nanoscale",
    "Nature",
    "Nature Biomedical Engineering",
    "Nature Cancer",
    "Nature Communications",
    "Nature Materials",
    "Nature Medicine",
    "Nature Nanotechnology",
    "NPG Asia Materials",
    "Pharmaceutics",
    "PNAS",
    "Science",
    "Science Advances",
    "Science Translational Medicine",
    "Scientific Reports",
    "Small"
]

# journals as small letters
journals_to_include = [journal.lower() for journal in journals_to_include]

# Inclusion criteria
keywords_inclusion = ["nanoparticles", "nanoparticle", "in vivo", "mouse model", "animal model"]

# Exclusion criteria
keywords_exclusion = ["review"]

In [8]:
# compile the query
query = f"({keywords_inclusion[0]} OR {keywords_inclusion[1]}) AND ({keywords_inclusion[2]} OR {keywords_inclusion[3]} OR {keywords_inclusion[4]})"

if True:
    # add the exclusion criteria
    query += f" AND NOT ({keywords_exclusion[0]})"

    # add the journals to include
    query += f" AND ("
    for journal in journals_to_include:
        query += f"source:{journal} OR "
    query = query[:-4]
    query += ")"



In [9]:
query

'(nanoparticles OR nanoparticle) AND (in vivo OR mouse model OR animal model) AND NOT (review) AND (source:acs applied materials & interfaces OR source:acs nano OR source:advanced functional materials OR source:advanced materials OR source:angewandte chemie OR source:biology and medicine OR source:biomaterials OR source:cell OR source:clinical cancer research OR source:frontiers in nanotechnology OR source:immunity OR source:international journal of nanomedicine OR source:journal of controlled release OR source:journal of materials chemistry b OR source:matter OR source:molecular therapy OR source:nano letters OR source:nano micro small OR source:nano research OR source:nanomedicine OR source:nanomedicine: nanotechnology OR source:nanoscale OR source:nature OR source:nature biomedical engineering OR source:nature cancer OR source:nature communications OR source:nature materials OR source:nature medicine OR source:nature nanotechnology OR source:npg asia materials OR source:pharmaceutic

In [10]:

# search for the query
record = search_pubmed(query)

# print the number of results
print(f"Number of results: {record['Count']}")

Number of results: 131


In [11]:

# get the ids
id_list = record['IdList']

# fetch the data
papers = fetch_pubmed_data_given_ids_in_chunks(id_list, output_path="papers")


